# Expectation value 

In [1]:
import numpy as np
from qiskit.circuit.library import IQP
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp, random_hermitian
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as Estimator

import keyring as k3


In [2]:
# Get the API token from keyring
api_token = k3.get_password('ibm_quantum', 'api_token')

In [ ]:
# set number of qubits for the IQP circuit, adjust as needed
n_qubits = 120  

In [5]:
# set up your backend 
service = QiskitRuntimeService(channel="ibm_quantum", token=api_token)
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=n_qubits
)

In [6]:
# Create your IQP circuit and observable
mat = np.real(random_hermitian(n_qubits, seed=1234))
circuit = IQP(mat)
observable = SparsePauliOp("Z" * n_qubits)  # Define the observable as Z^n_qubits for the expectation value
 
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(circuit)
isa_observable = observable.apply_layout(isa_circuit.layout)

In [8]:
# create the job and run it

estimator = Estimator(mode=backend)
job = estimator.run([(isa_circuit, isa_observable)])
job_id = job.job_id()
print(f" > Job ID: {job_id}")

 > Job ID: czy44hjqnmvg008w1n30


In [9]:
# Wait for the job to complete and get the result
result = job.result()
 
print(f" > Expectation value: {result[0].data.evs}")
print(f" > Metadata: {result[0].metadata}")

 > Expectation value: -0.9411764705882353
 > Metadata: {'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32}
